# Notebook 3: Exploring TB and Socio-economic Indicators, Part 2

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/NB03-World-TB-Notebook-Part-2.ipynb)



Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `NB03-World-TB-Notebook-Part-2.ipynb` at `6d50f558cd39b40089cd3eea1fbc19425e5191d1`; original SHA-256 `e96bd6aa4903952055500e5255685c85a3f5775ff426f648d6b0dac91718ce2c`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run ./resources/library.py

In [ ]:
style_notebook()

# Notebook 3: Exploring TB and Socio-economic Indicators, Part 2

## Review Goals

Our goal for this TB exploration notebook is to construct a "gapminder" for TB data and a time series choropleth map. See figures below.

[Source image continuation: Gapminder for TB](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/images/gapminder-for-tb.png)

**Figure 1**. Interactive Gapminder bubble chart for TB data

[Source image continuation: Choropleth Timeseries for TB](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/images/choropleth-timeseries-tb.png)

**Figure 2**. Interactive choropleth map for TB data

## Steps in this notebook

Notebook 3
3. Merge, clean, explore dataframes
4. Generate plotly express visualizations

## STEP 1 (Review). Set up data exploration environment

To create visualizations, we will use Plotly Express. You can learn more about Plotly Express from:
1. [Plotly GitHub Site](https://github.com/plotly/plotly_express) (GitHub)
2. [Jupyter Notebook Example](https://www.plotly.express/)
3. Some articles written about it [here](https://medium.com/@plotlygraphs/introducing-plotly-express-808df010143d) and [here](https://towardsdatascience.com/plotly-express-the-good-the-bad-and-the-ugly-dc941649687c)

In [ ]:
import plotly.express as px
import plotly

plotly.__version__

We will also use our Swiss Army knife for data science, `pandas`.

In [ ]:
import pandas as pd
pd.__version__

In [ ]:
pd.set_option('max_colwidth', 150)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

**`missingno`**: Missing data visualization module for Python 

Read more about it here: https://github.com/ResidentMario/missingno

In [ ]:
import missingno as mno
mno.__version__

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np

np.__version__

## STEP 2. Download and clean up data

Since we have created all the necessary dataframes in Notebook 2 and pickled these, we will just load the pickle files here. All data pickle files are found in the `outputs` folder.

Load TB data:

In [ ]:
tb_df2 = pd.read_pickle('outputs/tb_df2.pickle')
tb_dict_df = pd.read_pickle('outputs/tb_dict_df.pickle')

Load World Bank GDP, GDP per capita data, and country metadata:

In [ ]:
wb_gdp_df5 = pd.read_pickle('outputs/wb_gdp_df5.pickle')
wb_gdppc_df5 = pd.read_pickle('outputs/wb_gdppc_df5.pickle')
wb_gdp_meta_df3 = pd.read_pickle('outputs/wb_gdp_meta_df3.pickle')

Load World Bank education data (literacy rate, primary completion rate) and indicator metadata:

In [ ]:
wb_ed_df2 = pd.read_pickle('outputs/wb_ed_df2.pickle')
wb_edmeta_df1 = pd.read_pickle('outputs/wb_edmeta_df1.pickle')
wb_ed_primcomp_df4 = pd.read_pickle('outputs/wb_ed_primcomp_df4.pickle')

## STEP 3. Clean, merge, explore data sets

### GDP data

#### Merge #1: Merge WB GDP data and metadata dataframes on `ISO_Alpha`

In [ ]:
merged_df1 = \
    pd.merge(wb_gdp_meta_df3, wb_gdp_df5, on=['ISO_Alpha'], how='inner').\
    drop(columns=['Country_y'])

merged_df1.head()

In [ ]:
merged_df2 = merged_df1.rename(columns={'Country_x': 'Country'})

merged_df2.head()

#### Merge #2: Merge WB GDP and GDP per capita data

In [ ]:
merged_df3 = \
    pd.merge(wb_gdppc_df5, merged_df2, on=['ISO_Alpha','Year'], how='inner').\
    drop(columns=['Country_y'])

merged_df3

In [ ]:
merged_df4 = merged_df3.rename(columns={'Country_x': 'Country'})

merged_df4

In [ ]:
mno.bar(merged_df4)

In [ ]:
tb_nonmatch = np.setdiff1d(tb_df2.ISO_Alpha.unique(), merged_df4.ISO_Alpha.unique())

tb_nonmatch

It seems we have a few countries that did not match from the two dataframes. Let's take a closer look, and view the country names.

In [ ]:
for iso3 in tb_nonmatch:
    print(iso3, tb_df2.query("ISO_Alpha=='"+iso3+"'")['Country'].unique())

Do you know where these countries are?

#### Merge #3: Merge WB GDP, GDP per capita and primary completion rate

In [ ]:
merged_df5 = \
    pd.merge(wb_ed_primcomp_df4, merged_df4, on=['ISO_Alpha','Year'], how='inner').\
    drop(columns=['Country_y'])

merged_df5

In [ ]:
merged_df6 = \
    merged_df5.rename(\
        columns={'Country_x': 'Country','Value':'Primary-Completion-Rate'})

merged_df6

#### Merge #4: Merge TB data with WB GDP data

In [ ]:
merged_df7 = pd.merge(merged_df6, tb_df2, on=['ISO_Alpha', 'Year'], how='inner')
merged_df8 = merged_df7.rename(columns={'Country_x': 'Country'})

merged_df8.info()

In [ ]:
mno.bar(merged_df8)

In [ ]:
mno.heatmap(merged_df8)

### 2016 subset of second-merge dataframe

In [ ]:
is_2016 = merged_df8['Year'] == 2016

In [ ]:
merged_df8_2016 = merged_df8[is_2016]

In [ ]:
merged_df8_2016.info()

#### Pickle merged files

In [ ]:
merged_df8_2016.to_pickle('outputs/merged_df8_2016.pickle')
merged_df4.to_pickle('outputs/merged_df4.pickle')
merged_df8.to_pickle('outputs/merged_df8.pickle')

## You finished Notebook 3!

Please proceed to Notebook 4 to see the visualizations.